In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
BRONZE_TABLE = "workspace.bronze.orders"

SILVER_TABLE = "workspace.silver.orders"

QUARANTINE_TABLE = "workspace.silver.orders_quarantine"

bronze_df = spark.table(BRONZE_TABLE)

print("Bronze records:", bronze_df.count())

In [0]:
typed_df = (
    bronze_df

    .withColumn(
        "order_date_typed",
        F.expr("try_cast(order_date AS DATE)")
    )

    .withColumn(
        "quantity_typed",
        F.expr("try_cast(quantity AS INT)")
    )

    .withColumn(
        "unit_price_typed",
        F.expr("try_cast(unit_price AS DECIMAL(18,2))")
    )

    .withColumn(
        "discount_typed",
        F.expr("try_cast(discount AS DECIMAL(5,2))")
    )
)

In [0]:
display(
    typed_df.select(
        "order_id",
        "order_date",
        "order_date_typed",
        "quantity",
        "quantity_typed",
        "unit_price",
        "unit_price_typed",
        "discount",
        "discount_typed"
    ).limit(20)
)

In [0]:
validated_df = (
    typed_df

    .withColumn(
        "invalid_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )

    .withColumn(
        "invalid_customer",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )

    .withColumn(
        "invalid_product_name",
        F.col("product_name").isNull()
        | (F.trim(F.col("product_name")) == "")
    )

    .withColumn(
        "invalid_date",
        F.col("order_date_typed").isNull()
    )

    .withColumn(
        "invalid_quantity",
        F.col("quantity_typed").isNull()
        | (F.col("quantity_typed") <= 0)
    )

    .withColumn(
        "invalid_price",
        F.col("unit_price_typed").isNull()
        | (F.col("unit_price_typed") < 0)
    )

    .withColumn(
        "invalid_discount",
        F.col("discount_typed").isNull()
        | (F.col("discount_typed") < 0)
        | (F.col("discount_typed") > 100)
    )

    .withColumn(
        "invalid_status",
        ~F.col("order_status").isin(
            "Completed",
            "Cancelled",
            "Pending",
            "Returned"
        )
    )
)

In [0]:
duplicate_order_ids = (
    validated_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
    .withColumn("is_duplicate_order", F.lit(True))
)

validated_df = (
    validated_df
    .join(
        duplicate_order_ids,
        on="order_id",
        how="left"
    )
    .withColumn(
        "is_duplicate_order",
        F.coalesce(
            F.col("is_duplicate_order"),
            F.lit(False)
        )
    )
)

validated_df = (
    validated_df

    .withColumn(
        "validation_error",
        F.concat_ws(
            "; ",
            F.when(
                F.col("invalid_order_id"),
                F.lit("NULL_OR_EMPTY_ORDER_ID")
            ),
            F.when(
                F.col("invalid_customer"),
                F.lit("NULL_OR_EMPTY_CUSTOMER_ID")
            ),
            F.when(
                F.col("invalid_product_name"),
                F.lit("NULL_OR_EMPTY_PRODUCT_NAME")
            ),
            F.when(
                F.col("invalid_date"),
                F.lit("INVALID_ORDER_DATE")
            ),
            F.when(
                F.col("invalid_quantity"),
                F.lit("INVALID_QUANTITY")
            ),
            F.when(
                F.col("invalid_price"),
                F.lit("INVALID_UNIT_PRICE")
            ),
            F.when(
                F.col("invalid_discount"),
                F.lit("INVALID_DISCOUNT")
            ),
            F.when(
                F.col("invalid_status"),
                F.lit("INVALID_ORDER_STATUS")
            ),
            F.when(
                F.col("is_duplicate_order"),
                F.lit("DUPLICATE_ORDER_ID")
            )
        )
    )
)
validated_df = (
    validated_df
    .withColumn(
        "is_valid",
        ~(
            F.col("invalid_order_id")
            | F.col("invalid_customer")
            | F.col("invalid_product_name")
            | F.col("invalid_date")
            | F.col("invalid_quantity")
            | F.col("invalid_price")
            | F.col("invalid_discount")
            | F.col("invalid_status")
            | F.col("is_duplicate_order")
        )
    )
)

display(
    validated_df.groupBy("is_valid").count()
)

In [0]:
silver_df = (
    validated_df
    .filter(F.col("is_valid"))

    .select(
        "order_id",
        "customer_id",
        "order_date_typed",
        "product_id",
        "product_name",
        "category",
        "quantity_typed",
        "unit_price_typed",
        "discount_typed",
        "country",
        "payment_method",
        "order_status",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )

    .withColumnRenamed(
        "order_date_typed",
        "order_date"
    )

    .withColumnRenamed(
        "quantity_typed",
        "quantity"
    )

    .withColumnRenamed(
        "unit_price_typed",
        "unit_price"
    )

    .withColumnRenamed(
        "discount_typed",
        "discount"
    )
)

display(silver_df.limit(20))

In [0]:
silver_df.printSchema()

In [0]:
quarantine_df = (
    validated_df
    .filter(~F.col("is_valid"))

    .select(
        "order_id",
        "customer_id",
        "order_date",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        "discount",
        "country",
        "payment_method",
        "order_status",
        "validation_error",
        "source_file",
        "batch_id",
        "ingestion_timestamp"
    )

    .withColumn(
        "rejection_timestamp",
        F.current_timestamp()
    )
)

In [0]:
display(quarantine_df.limit(20))

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS silver_records
FROM {SILVER_TABLE}
""").show()

In [0]:
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(QUARANTINE_TABLE)
)

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS quarantined_records
FROM {QUARANTINE_TABLE}
""").show()

In [0]:
display(
    spark.sql("""
        SELECT 'Bronze' AS layer,
               COUNT(*) AS records
        FROM workspace.bronze.orders

        UNION ALL

        SELECT 'Silver',
               COUNT(*)
        FROM workspace.silver.orders

        UNION ALL

        SELECT 'Quarantine',
               COUNT(*)
        FROM workspace.silver.orders_quarantine
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            validation_error,
            COUNT(*) AS record_count
        FROM workspace.silver.orders_quarantine
        GROUP BY validation_error
        ORDER BY record_count DESC
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_records,

            SUM(
                CASE
                    WHEN customer_id IS NULL THEN 1
                    ELSE 0
                END
            ) AS null_customers,

            SUM(
                CASE
                    WHEN quantity <= 0 THEN 1
                    ELSE 0
                END
            ) AS invalid_quantities,

            SUM(
                CASE
                    WHEN unit_price < 0 THEN 1
                    ELSE 0
                END
            ) AS invalid_prices,

            SUM(
                CASE
                    WHEN discount < 0 OR discount > 100 THEN 1
                    ELSE 0
                END
            ) AS invalid_discounts

        FROM workspace.silver.orders
    """)
)